# Story and Image Generator Agent
Uses GPT-4o-mini for text processing and DALL-E 2 for image generation via smolagents.

## Setup and Imports

In [ ]:
from pathlib import Path
from smolagents import CodeAgent, LiteLLMModel, tool
import os
import re
from openai import OpenAI
import requests
from datetime import datetime

## Environment Setup

In [ ]:
# Set your OpenAI API key
os.environ['OPENAI_API_KEY'] = 'your-openai-api-key-here'

assert os.getenv('OPENAI_API_KEY') and os.getenv('OPENAI_API_KEY') != 'your-openai-api-key-here', \
    'Set OPENAI_API_KEY before running.'

## Initialize Model
Must be defined before tools that reference it.

In [ ]:
# Initialize the LLM model (used by tools below)
model = LiteLLMModel(model_id="gpt-4o-mini")

## Load the Story

In [ ]:
CHAPTER_PATH = "alice_in_wonderland.txt"
OUTPUT_MD = "image_story_output.md"
IMAGE_OUTPUT_DIR = "generated_images"

chapter_text = Path(CHAPTER_PATH).read_text(encoding="utf-8")
print(f"Loaded {len(chapter_text.split())} words from {CHAPTER_PATH}")

## Tool 1: Text Condensation

In [ ]:
@tool
def text_condensation_tool(text: str, target_sentences: int = 4) -> str:
    """
    Condenses a text segment into a target number of sentences
    while preserving narrative flow and key story beats.

    Args:
        text: The full text segment to condense
        target_sentences: Target number of sentences (default 4)

    Returns:
        Condensed narrative text
    """
    prompt = (
        f"Condense the following story segment into {target_sentences} clear, "
        f"narrative sentences. Capture key story beats, keep character names and "
        f"important actions, write in past tense active voice.\n\n"
        f"Text:\n{text}\n\nCondensed narrative:"
    )

    messages = [{"role": "user", "content": prompt}]
    response = model(messages, stop_sequences=[])

    condensed = response.content if hasattr(response, 'content') else str(response)
    condensed = condensed.strip().strip('"')
    condensed = re.sub(r'^(Condensed narrative:|Summary:)\s*', '', condensed, flags=re.IGNORECASE)
    return condensed

## Tool 2: Image Prompt Generation (Verbal Sampling)
Generates N candidate prompts with different creative lenses, then merges the best elements.

In [ ]:
def _generate_single_candidate(summary_text: str, raw_segment: str, style_hint: str) -> str:
    """Generate one candidate image prompt with a given style emphasis."""
    prompt = (
        f"Create a detailed image prompt for an illustrator based on this story segment.\n\n"
        f"SUMMARY:\n{summary_text}\n\n"
        f"ORIGINAL TEXT (extract visual details from this):\n{raw_segment}\n\n"
        f"Style emphasis: {style_hint}\n\n"
        f"Include: scene description, character details (appearance, clothing, expressions, poses), "
        f"important objects/props, environment/setting, color palette and lighting, "
        f"mood/atmosphere, composition and framing.\n\n"
        f"Be specific and vivid. Describe what the viewer SEES, not abstract concepts.\n\n"
        f"Image prompt:"
    )
    messages = [{"role": "user", "content": prompt}]
    response = model(messages, stop_sequences=[])
    result = response.content if hasattr(response, 'content') else str(response)
    return result.strip()


def _select_and_merge(candidates: list, summary_text: str) -> str:
    """Use the LLM to pick the best elements from all candidates and merge them."""
    candidates_block = ""
    for i, c in enumerate(candidates, 1):
        candidates_block += f"\n--- CANDIDATE {i} ---\n{c}\n"

    merge_prompt = (
        f"You are selecting the best image generation prompt from multiple candidates.\n\n"
        f"STORY CONTEXT:\n{summary_text}\n\n"
        f"CANDIDATES:{candidates_block}\n\n"
        f"Your task:\n"
        f"1. Identify the most vivid and specific visual details from each candidate.\n"
        f"2. Merge the best elements into ONE final prompt.\n"
        f"3. Prioritize: concrete visual details > abstract descriptions, "
        f"specific colors/poses/expressions > vague mood words.\n"
        f"4. Remove redundancy. Keep it under 800 characters.\n"
        f"5. Output ONLY the final scene description, no headers or labels.\n\n"
        f"Final merged image prompt:"
    )
    messages = [{"role": "user", "content": merge_prompt}]
    response = model(messages, stop_sequences=[])
    result = response.content if hasattr(response, 'content') else str(response)
    return result.strip()


@tool
def image_prompt_generation_tool(summary_text: str, raw_segment: str, n_samples: int = 3) -> str:
    """
    Creates a detailed image prompt using verbal sampling: generates
    multiple candidate prompts with different creative lenses, then
    selects and merges the best visual details into one final prompt.

    Args:
        summary_text: The narrative summary (what happens in the scene)
        raw_segment: The original story text (for extracting visual details)
        n_samples: Number of candidate prompts to generate (default 3)

    Returns:
        A detailed, merged image generation prompt
    """
    # Each candidate gets a different creative lens to maximize diversity
    style_hints = [
        "Focus on character expressions, body language, and emotional tension.",
        "Focus on environment, lighting, color palette, and atmospheric details.",
        "Focus on composition, dynamic action, object placement, and cinematic framing.",
        "Focus on textures, small props, background details, and visual storytelling cues.",
    ]

    # Generate N diverse candidates
    candidates = []
    for i in range(min(n_samples, len(style_hints))):
        candidate = _generate_single_candidate(summary_text, raw_segment, style_hints[i])
        candidates.append(candidate)
        print(f'  Generated candidate {i+1}/{n_samples} ({len(candidate)} chars)')

    # If only 1 candidate, return directly
    if len(candidates) == 1:
        return candidates[0]

    # Merge the best elements from all candidates
    merged = _select_and_merge(candidates, summary_text)
    print(f'  Merged prompt: {len(merged)} chars')
    return merged


## Tool 3: Image Generation (DALL-E 2)

In [ ]:
def _smart_truncate_prompt(prompt: str, max_length: int = 1000) -> str:
    """Truncates a long prompt while preserving key visual sections."""
    sections = [
        (r"Scene description.*?(?=\n[A-Z]|\n\n|$)", "Scene: "),
        (r"Character details:.*?(?=\n[A-Z]|\n\n|$)", "Characters: "),
        (r"Important objects.*?(?=\n[A-Z]|\n\n|$)", "Objects: "),
        (r"Environment/setting.*?(?=\n[A-Z]|\n\n|$)", "Setting: "),
        (r"Color and lighting.*?(?=\n[A-Z]|\n\n|$)", "Lighting: "),
        (r"Mood and atmosphere.*?(?=\n[A-Z]|\n\n|$)", "Mood: "),
    ]

    result_parts = []
    remaining = max_length

    for pattern, prefix in sections:
        match = re.search(pattern, prompt, re.DOTALL | re.IGNORECASE)
        if match and remaining > 0:
            content = re.sub(r'\n\s*[-\u2022]\s*', ' ', match.group(0))
            content = re.sub(r'\s+', ' ', content).strip()
            for label in ['Character details:', 'Important objects:']:
                content = content.replace(label, '')
            entry = prefix + content.strip()
            if len(entry) <= remaining:
                result_parts.append(entry)
                remaining -= len(entry) + 2
            else:
                avail = remaining - len(prefix) - 3
                if avail > 20:
                    result_parts.append(prefix + content[:avail] + '...')
                break

    return '. '.join(result_parts) if result_parts else prompt[:max_length - 3] + '...'


@tool
def image_generation_tool(
    image_prompt: str,
    page_number: int,
    output_dir: str = "generated_images"
) -> str:
    """
    Generates an image from a text prompt using DALL-E 2.

    Args:
        image_prompt: Detailed description of the image to generate
        page_number: Page number for naming the output file
        output_dir: Directory to save generated images

    Returns:
        File path to the saved image
    """
    api_key = os.getenv('OPENAI_API_KEY')
    if not api_key:
        raise ValueError('OPENAI_API_KEY not set')

    client = OpenAI(api_key=api_key)
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    # DALL-E 2 has a 1000-char prompt limit
    if len(image_prompt) > 1000:
        image_prompt = _smart_truncate_prompt(image_prompt, max_length=1000)

    response = client.images.generate(
        model='dall-e-2',
        prompt=image_prompt,
        size='512x512',
        quality='standard',
        n=1,
    )

    image_url = response.data[0].url
    image_data = requests.get(image_url)
    image_data.raise_for_status()

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filepath = output_path / f'page_{page_number:02d}_{timestamp}.png'

    with open(filepath, 'wb') as f:
        f.write(image_data.content)

    print(f'Image saved: {filepath.name}')
    return str(filepath.resolve())

## Create Agent

In [ ]:
agent = CodeAgent(
    tools=[text_condensation_tool, image_prompt_generation_tool, image_generation_tool],
    model=model,
    max_steps=50,
    verbosity_level=1
)
print('Agent initialized.')

## Run the Pipeline

In [ ]:
task = f"""
Convert the story into a visual story format with multiple pages.
Each page needs: STORY summary, IMAGE PROMPT, and GENERATED IMAGE.

Story: {chapter_text}

Instructions:
1. Split into segments at natural story beats
2. For each segment:
   - Use text_condensation_tool(segment, target_sentences=4)
   - Use image_prompt_generation_tool(condensed_story, segment)
   - Use image_generation_tool(image_prompt, page_number, \"{IMAGE_OUTPUT_DIR}\")
3. Output format:

## PAGE 1
STORY
<condensed text>

IMAGE PROMPT
<image description>

IMAGE FILE
<path to generated image>

Call final_answer(final_markdown) when done.
"""

result_markdown = agent.run(task)

## Save Results

In [ ]:
out_path = Path(OUTPUT_MD)
out_path.write_text(str(result_markdown), encoding='utf-8')
print(f'Markdown saved to: {out_path.resolve()}')

image_dir = Path(IMAGE_OUTPUT_DIR)
if image_dir.exists():
    images = list(image_dir.glob('*.png'))
    print(f'Total images generated: {len(images)}')

# Preview
print('\nPreview (first 30 lines):')
print('\n'.join(str(result_markdown).splitlines()[:30]))

## View Generated Images

In [ ]:
from IPython.display import Image, display
import glob

image_files = sorted(glob.glob(f'{IMAGE_OUTPUT_DIR}/*.png'))
for img_path in image_files:
    print(Path(img_path).name)
    display(Image(filename=img_path))

if not image_files:
    print('No images found.')